# Loading

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import BertTokenizerFast, BertModel
from datasets import load_dataset, Dataset
from hierarchical_binary_tree import HierarchicalBinaryTree
import torch.optim as optim
from tqdm import tqdm
import random
import torch.nn.functional as F
import numpy as np
import json

In [2]:
# Hyperparameters

model_name = 'bert-base-cased' # teacher model
chunk_size = 64
max_seq_length = 512 # matching teacher model
batch_size = 8
num_epochs = 1 # don't want to overfit
learning_rate = 2e-5
weight_decay = 0.01
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
tokenizer = BertTokenizerFast.from_pretrained(model_name)


## Messing with FriendsQA

In [ ]:
# question_token = "[Q]"
# context_token = "[S]"
# # speaker_token = "[SPK]"
# special_tokens_dict = {
#     "additional_special_tokens": [question_token, context_token]
# }
# tokenizer.add_special_tokens(special_tokens_dict) #, replace_additional_special_tokens=False)

2

In [4]:
BLACKLISTED_QAS = [
    {
        'title': 's04_e04_c10',
        'qid': 's04_e04_c10_Why',
        'aid': 1
    },
    {
        'title': 's04_e04_c10',
        'qid': 's04_e04_c10_Why_Paraphrased',
        'aid': 1
    },
]


In [5]:
def load_friendsqa(filepath):
  all_data = []
  with open(filepath, 'r') as file:
    friendsqa_data = json.load(file)['data']
  for scene in friendsqa_data:
    assert len(scene['paragraphs']) == 1
    all_data += preprocess_friendsqa(scene['paragraphs'][0], scene['title'])
  return all_data


In [6]:
def get_scene_idx(ans, paragraph, paragraph_lines):
  real_answer = ans['answer_text']
  ans_sent = ans['utterance_id']
  ans_start = ans['inner_start']
  ans_end = ans['inner_end']
  abs_start = 0
  for i in range(ans_sent):
      abs_start += len(paragraph_lines[i]) + 2 # '\n '
  line = paragraph_lines[ans_sent]
  colon_index = line.index(':')
  if not ans['is_speaker']:
    abs_start += colon_index + 2 # ': '
    utterance_part = line[colon_index+2:]
    utterance_words = utterance_part.split(' ')
    abs_start += sum(len(w) + 1 for w in utterance_words[:ans_start])
  abs_end = abs_start + len(real_answer)
  if paragraph[abs_start:abs_end] != real_answer:
    print(ans)
    print(paragraph)
    print('estimated:', paragraph[abs_start:abs_end])
  assert paragraph[abs_start:abs_end] == real_answer
  return abs_start, abs_end

In [7]:
def valid_question(title, q_id, a_id):
    for banned_qa in BLACKLISTED_QAS:
        if title == banned_qa['title'] and q_id == banned_qa['qid'] and a_id == banned_qa['aid']:
            return False
    return True

In [8]:
def preprocess_friendsqa(paragraph_dict, paragraph_title):
  utterances = paragraph_dict['utterances:']
  paragraph_lines = [': '.join([','.join(u['speakers']), u['utterance']]) for u in utterances]
  paragraph = '\n '.join(paragraph_lines)
  qas = []
  for question in paragraph_dict['qas']:
    for i, ans in enumerate(question['answers']):
      if not valid_question(paragraph_title, question['id'], i):
        continue

      abs_start, abs_end = get_scene_idx(ans, paragraph, paragraph_lines)

      qas.append({
          'scene': paragraph,
          'question': question['question'],
          'answer_start': abs_start,
          'answer_end': abs_end,
          'title': paragraph_title,
          'question_id': question['id'],
          'answer_id': i
      })

  return qas


In [9]:
train_data = load_friendsqa('../datasets_/FriendsQA/dat/friendsqa_trn.json')
val_data = load_friendsqa('../datasets_/FriendsQA/dat/friendsqa_dev.json')
test_data = load_friendsqa('../datasets_/FriendsQA/dat/friendsqa_tst.json')

In [10]:
train_dataset = Dataset.from_list(train_data)

In [11]:
def encode_data(cqa_dict): # using global tokenizer and max_seq_length
    question = cqa_dict['question']
    scene = cqa_dict['scene']
    abs_start = cqa_dict['answer_start']
    abs_end = cqa_dict['answer_end']
    stride = (max_seq_length - len(tokenizer(question))) // 2

    encoding = tokenizer(
        question,
        scene,
        truncation="only_second",
        padding="max_length",
        max_length=max_seq_length,
        stride=stride,
        return_offsets_mapping=True,
        return_overflowing_tokens=True,
        return_token_type_ids=True,
        return_tensors="pt",
    )

    input_ids = encoding["input_ids"]
    offset_mappings = encoding["offset_mapping"]
    token_type_ids = encoding["token_type_ids"]

    for i in range(len(input_ids)):
        offsets = offset_mappings[i].tolist()
        token_types = token_type_ids[i].tolist()

        answer_token_start = None
        answer_token_end = None

        for idx, (start_char, end_char) in enumerate(offsets):
            if token_types[idx] != 1:  # skip question + special tokens
                continue
            if answer_token_start is None and start_char <= abs_start < end_char:
                answer_token_start = idx
            if start_char < abs_end <= end_char:
                answer_token_end = idx
                break

        # Backup: include entire span
        if answer_token_start is not None and answer_token_end is None:
            for idx in range(answer_token_start + 1, len(offsets)):
                s, e = offsets[idx]
                if token_types[idx] != 1:
                    continue
                if s >= abs_end:
                    answer_token_end = idx - 1
                    break
            if answer_token_end is None:
                answer_token_end = len(offsets) - 1

        if answer_token_start is not None and answer_token_end is not None:
            return {
                "input_ids": input_ids[i],
                "attention_mask": encoding["attention_mask"][i],
                "token_type_ids": token_type_ids[i],
                "answer_token_start": answer_token_start,
                "answer_token_end": answer_token_end,
                "offset_mapping": offset_mappings[i],
            }
    return {} # Allegedly removed automatically by dataset.map operation

In [12]:
def preprocess_dataset(data, batch_size, shuffle):
    '''
    Args:
    data: list of dicts containing keys ['scene', 'question', 'answer_start', 'answer_end', 'title', 'question_id', 'answer_id']
    
    Return:
    DataLoader object of this encoded dataset
    '''
    dataset = Dataset.from_list(data)
    preprocessed = dataset.map(encode_data)
    loader = DataLoader(preprocessed, batch_size=batch_size, shuffle=shuffle)
    return loader

In [ ]:
# preprocessed_train = train_dataset.map(encode_data)

Map:   0%|          | 0/16660 [00:00<?, ? examples/s]

In [ ]:
# print(len(preprocessed_train))
# preprocessed_train = preprocessed_train.filter(lambda example: example is not None)
# print(len(preprocessed_train))

16660


Filter:   0%|          | 0/16660 [00:00<?, ? examples/s]

16660


In [ ]:
# train_loader = DataLoader(preprocessed_train, batch_size=batch_size, shuffle=True)

In [16]:
train_loader = preprocess_dataset(train_data, batch_size, True)
val_loader = preprocess_dataset(val_data, batch_size, False)
test_loader = preprocess_dataset(test_data, batch_size, False)

Map:   0%|          | 0/16660 [00:00<?, ? examples/s]

Map:   0%|          | 0/1942 [00:00<?, ? examples/s]

Map:   0%|          | 0/2105 [00:00<?, ? examples/s]

In [ ]:
# test_instance = {'scene': "#NOTE#: [ Scene : Rachel 's Bedroom , Ross is finishing up her make - up . ]\n Ross Geller: There you go ! Good enough for your party , huh ?\n #NOTE#: ( She turns and looks in the mirror , and it 's way , way over done . She looks like she has two black eyes . )\n Rachel Green: Sure .\n Ross Geller: Yep ?\n Rachel Green: Sure , I 'll just sit next to the trans-sexual from purchasing .\n Ross Geller: Okay , come on ! All right , I got ta go ! So good luck at the party . Okay ?\n Rachel Green: Oh wait , Ross , would you just stay and help me get dressed ?\n Ross Geller: Sure , okay .\n Rachel Green: Okay . Okay , great ! Umm , okay , just turn around .\n Ross Geller: What ?\n Rachel Green: I do n't want you to see me naked !\n Ross Geller: Rachel , I 've seen you naked a million times . I ate hot fudge off you naked . Remember , I - I sucked that mini-marshmallow out of your belly button ?\n Rachel Green: Yeah , but that was different . Y'know ? I mean , we were , we were going out then , now I think it 's weird .\n Ross Geller: Rach , y'know I can see you naked any time I want .\n Rachel Green: What ?\n Ross Geller: All I have to do is close my eyes . See ? Woo - hoo !!\n Rachel Green: Ross ! Stop that !\n Ross Geller: Ah , I 'm sorry .\n Rachel Green: Come on ! I do n't want you thinking of me like that any more !\n Ross Geller: Ahh , sorry , nothing you can do about it . It 's one of my ah , rights as the ex-boyfriend . Oop , oh yeah !\n Rachel Green: Stop it ! Cut it out ! Cut it out !\n Ross Geller: Okay , okay , I 'm sorry , it will never happen ... Uh-oh ! Wait a minute ! Wait - wait , now there are a hundred of you and I 'm the king .\n Rachel Green: Rosss ...\n Ross Geller: Come on , would you grow up ? It 's no big deal .\n Rachel Green: All right . Fine .\n Ross Geller: Yowzah !!!\n Rachel Green: O-kay !! See what you did , I 'm gon na be doing it by myself now . Okay ?\n Ross Geller: Aww , come on .\n Rachel Green: That 's it . Ow !!!\n Ross Geller: Oh my God !\n Rachel Green: Oh - ow !\n Ross Geller: All right .\n Rachel Green: Ow !\n Ross Geller: Look ...\n Rachel Green: Ow !\n Ross Geller: Okay .\n Rachel Green: Ow !\n Ross Geller: Rach ?\n Rachel Green: Ow ! Ow !\n Ross Geller: Easy . Easy . You have to go to the hospital . Okay ?\n Rachel Green: Okay , I do .\n Ross Geller: Okay .\n Rachel Green: I really do .\n Ross Geller: Okay , I 'm gon na get your coat and then I 'll - I 'll put you in a cab .\n Rachel Green: Okay . Oh wait , wait - wait , you 're not gon na come with me ?\n Ross Geller: Of course I am . I just have to make a call .\n Rachel Green: Okay .\n Ross Geller: Okay ?\n Rachel Green: Thank you . Oww !!!! God !\n Ross Geller: What ?! I wh - , what 's wrong ?\n Rachel Green: I 'm sorry , I just ca n't go to the hospital lookin ' like this .",
#  'question': "How ca n't Rachel go to the hospital ?",
#  'answer_start': 2747,
#  'answer_end': 2765}

In [ ]:
# Teacher Model (Frozen BERT)
teacher_model = BertModel.from_pretrained(model_name, output_attentions=True).to(device)
teacher_model.eval()
for p in teacher_model.parameters():
    p.requires_grad = False

In [ ]:
# Student Model

model = HierarchicalBinaryTree(
    model_name=model_name,
    chunk_size=chunk_size
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

# Training

In [ ]:
# Training Loop - 1% of the training data

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    progress = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    for i, batch in enumerate(progress):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        label_pos = batch['label_pos'].to(device)

        with torch.no_grad():
            teacher_outputs = teacher_model(input_ids=input_ids, attention_mask=attention_mask)
            teacher_attn = torch.stack(teacher_outputs.attentions, dim=1)  # (B, Layers, Heads, T, T)
            teacher_attn = teacher_attn[:, -1, :, 0]  # use last layer, CLS as query
            teacher_attn = teacher_attn.mean(dim=1)  # avg over heads → (B, T)

        loss = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            label_pos=label_pos,
            teacher_attn=teacher_attn.unsqueeze(1)  # (B, 1, T)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        print("Batch Loss:", loss.item())
        progress.set_postfix(loss=total_loss / (progress.n + 1))

    print(f"Epoch {epoch+1} average loss: {total_loss / len(train_loader):.4f}")


Epoch 1:   1%|          | 1/110 [00:21<38:29, 21.19s/it, loss=76.6]

Batch Loss: 76.55315399169922


Epoch 1:   2%|▏         | 2/110 [00:44<40:29, 22.50s/it, loss=80.2]

Batch Loss: 83.89768981933594


Epoch 1:   3%|▎         | 3/110 [01:05<39:02, 21.89s/it, loss=80.1]

...

Batch Loss: 15.67839241027832


Epoch 1:  99%|█████████▉| 109/110 [48:56<00:22, 22.97s/it, loss=21.4]

Batch Loss: 16.48655128479004


Epoch 1: 100%|██████████| 110/110 [49:09<00:00, 26.82s/it, loss=21.3]

Batch Loss: 15.006429672241211
Epoch 1 average loss: 21.3273


In [ ]:
len(progress)

110

In [ ]:
checkpoint_path = "checkpoint.pth"

torch.save({
    'batch': i,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    # add other info if you want, e.g. loss or scheduler state
}, checkpoint_path)

print(f"Checkpoint saved to {checkpoint_path}")


Checkpoint saved to checkpoint.pth


In [ ]:
# Training Loop - now with all the data

training_progress = "batch_losses.csv"

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    progress = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    for i, batch in enumerate(progress):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        label_pos = batch['label_pos'].to(device)

        with torch.no_grad():
            teacher_outputs = teacher_model(input_ids=input_ids, attention_mask=attention_mask)
            teacher_attn = torch.stack(teacher_outputs.attentions, dim=1)  # (B, Layers, Heads, T, T)
            teacher_attn = teacher_attn[:, -1, :, 0]  # use last layer, CLS as query
            teacher_attn = teacher_attn.mean(dim=1)  # avg over heads → (B, T)

        loss = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            label_pos=label_pos,
            teacher_attn=teacher_attn.unsqueeze(1)  # (B, 1, T)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # print("Batch Loss:", loss.item())
        with open(training_progress, 'a') as f:
            f.write(f"{loss.item()}\n")
        progress.set_postfix(loss=total_loss / (progress.n + 1))

    print(f"Epoch {epoch+1} average loss: {total_loss / len(train_loader):.4f}")

    checkpoint_path = f"checkpoint_{epoch}_1.pth"
    torch.save({
        'batch': i,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        # add other info if you want, e.g. loss or scheduler state
    }, checkpoint_path)

    print(f"Checkpoint saved to {checkpoint_path}")



Epoch 1:  43%|████▎     | 4690/10953 [33:57:45<45:21:13, 26.07s/it, loss=15.3]   


KeyboardInterrupt: 

In [ ]:
# Saved when I stopped training after almost 1000 minutes; then I restarted training loop above with same model and hyperparams
checkpoint_path = f"checkpoint_{epoch}.pth"
torch.save({
    'batch': i,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    # add other info if you want, e.g. loss or scheduler state
}, checkpoint_path)

print(f"Checkpoint saved to {checkpoint_path}")

Checkpoint saved to checkpoint_0.pth


In [ ]:
checkpoint_path = f"checkpoint_{epoch}_1.pth"
torch.save({
    'batch': i,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    # add other info if you want, e.g. loss or scheduler state
}, checkpoint_path)

print(f"Checkpoint saved to {checkpoint_path} after second round of training reached {i} batches")

Checkpoint saved to checkpoint_0_1.pth after second round of training reached 4690 batches


# Evaluation

In [ ]:
# Load checkpoint, then extract the model state_dict
checkpoint = torch.load('./checkpoint_0_1.pth', map_location=device)

# Make sure you load only the model state_dict
model_state_dict = checkpoint['model_state_dict']

model.load_state_dict(model_state_dict)
model.eval()
model.to(device)

HierarchicalBinaryTree(
  (encoder): LeafEncoder(
    (bert): BertModel(
      (embeddings): BertEmbeddings(
        (word_embeddings): Embedding(28996, 768, padding_idx=0)
        (position_embeddings): Embedding(512, 768)
        (token_type_embeddings): Embedding(2, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): BertEncoder(
        (layer): ModuleList(
          (0-11): 12 x BertLayer(
            (attention): BertAttention(
              (self): BertSdpaSelfAttention(
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Linear(in_features=768, out_features=768, bias=True)
                (value): Linear(in_features=768, out_features=768, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): BertSelfOutput(
                (dense): Linear(in_features=768, out_features=768, bia

In [ ]:
test_data = load_dataset("trivia_qa", "unfiltered", split="test")

README.md:   0%|          | 0.00/26.7k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/47 [00:00<?, ?it/s]

train-00000-of-00047.parquet:   0%|          | 0.00/215M [00:00<?, ?B/s]

train-00001-of-00047.parquet:   0%|          | 0.00/279M [00:00<?, ?B/s]

...

test-00004-of-00006.parquet:   0%|          | 0.00/295M [00:00<?, ?B/s]

test-00005-of-00006.parquet:   0%|          | 0.00/251M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87622 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11313 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10832 [00:00<?, ? examples/s]

In [ ]:
preprocessed_test_data = test_data.map(preprocess)
preprocessed_test_data.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label_start', 'label_end'])

test_loader = DataLoader(preprocessed_test_data, batch_size=batch_size, shuffle=False)

In [ ]:
def evaluate_attention_alignment(model, teacher_model, dataloader, device):
    model.eval()
    teacher_model.eval()
    kl_losses = []
    cosine_sims = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            label_pos = batch['label_pos'].to(device)

            # Teacher attention: last layer, [CLS] query, averaged across heads
            teacher_outputs = teacher_model(input_ids=input_ids, attention_mask=attention_mask, output_attentions=True)
            teacher_attn = torch.stack(teacher_outputs.attentions, dim=1)[:, -1, :, 0]  # (B, Heads, T)
            teacher_attn = teacher_attn.mean(dim=1)  # (B, T)

            # Student tree attention
            x, _ = model.encoder(input_ids=input_ids, attention_mask=attention_mask)
            # x is last_hidden_state, disregarding the attentions
            query = x[:, 0, :]
            tree_attn, _ = model.tree_model(x, query, label_pos) # disregarding here per level sample loss

            # KL and cosine similarity
            kl = F.kl_div((tree_attn + 1e-8).log(), teacher_attn, reduction='batchmean')
            cos = F.cosine_similarity(tree_attn, teacher_attn, dim=-1).mean()

            kl_losses.append(kl.item())
            cosine_sims.append(cos.item())

    print(np.mean(kl_losses), np.mean(cosine_sims)) # to see if it's the same as the sum/len gpt did
    return {
        'kl_div': sum(kl_losses) / len(kl_losses),
        'cosine_similarity': sum(cosine_sims) / len(cosine_sims)
    }



In [ ]:
evaluate_attention_alignment(model, teacher_model, test_loader, device)

BertSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


11.990185200199594 0.3979227844202008


{'kl_div': 11.990185200199594, 'cosine_similarity': 0.3979227844202008}

In [ ]:
# def evaluate_chunk_accuracy(model, tokenizer, dataloader, device):
#     model.eval()
#     correct = 0
#     total = 0

#     with torch.no_grad():
#         for batch in dataloader:
#             input_ids = batch['input_ids'].to(device)
#             attention_mask = batch['attention_mask'].to(device)
#             label_pos = batch['label_pos'].to(device)

#             x, _ = model.encoder(input_ids=input_ids, attention_mask=attention_mask)
#             query = x[:, 0, :]
#             tree_attn, _ = model.tree_model(x, query, label_pos)

#             pred_idx = torch.argmax(tree_attn, dim=-1)  # (B,)
#             correct += (pred_idx == label_pos).sum().item()
#             total += input_ids.size(0)

#     accuracy = correct / total
#     return {'token_retrieval_accuracy': accuracy}


In [ ]:
# evaluate_chunk_accuracy(model, tokenizer, test_loader, device)

{'token_retrieval_accuracy': 0.9958456425406204}

In [ ]:
def debug_attention_mismatches(model, teacher_model, tokenizer, dataloader, device, n=1):
    model.eval()
    teacher_model.eval()

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            label_pos = batch['label_pos'].to(device)

            teacher_outputs = teacher_model(input_ids=input_ids, attention_mask=attention_mask, output_attentions=True)
            teacher_attn = torch.stack(teacher_outputs.attentions, dim=1)[:, -1, :, 0].mean(dim=1)  # (B, T)
            x, _ = model.encoder(input_ids=input_ids, attention_mask=attention_mask)
            query = x[:, 0, :]
            tree_attn, _ = model.tree_model(x, query, label_pos)

            for i in range(min(n, input_ids.size(0))):
                tokens = tokenizer.convert_ids_to_tokens(input_ids[i])
                tree_top = torch.argmax(tree_attn[i]).item()
                teacher_top = torch.argmax(teacher_attn[i]).item()
                gold = label_pos[i].item()

                print(f"\nExample {i+1}")
                print(f"Predicted token: {tokens[tree_top]} | Teacher token: {tokens[teacher_top]} | Gold: {tokens[gold]}")
                print(f"Input: {' '.join(tokens)}")


In [ ]:
debug_attention_mismatches(model, teacher_model, tokenizer, test_loader, device)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
def evaluate_full_answer_span(model, dataloader, tokenizer, device):
    model.eval()

    total_em = 0  # Exact Match score
    total_f1 = 0  # F1 score
    total_samples = 0

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            start_positions = batch['label_start'].to(device)  # ground truth start position
            end_positions = batch['label_end'].to(device)  # ground truth end position

            # Get model predictions
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            start_logits, end_logits = outputs.start_logits, outputs.end_logits

            # Convert logits to predicted positions
            start_pred = torch.argmax(start_logits, dim=-1)
            end_pred = torch.argmax(end_logits, dim=-1)

            # Calculate Exact Match (EM)
            em = ((start_pred == start_positions) & (end_pred == end_positions)).sum().item()
            total_em += em

            # Calculate F1 score
            # For each example, we need to compare the predicted and true spans
            for i in range(start_pred.size(0)):
                pred_start = start_pred[i].item()
                pred_end = end_pred[i].item()
                true_start = start_positions[i].item()
                true_end = end_positions[i].item()

                # F1 score is calculated based on the overlap between predicted and true spans
                pred_span = set(range(pred_start, pred_end + 1))
                true_span = set(range(true_start, true_end + 1))

                overlap = len(pred_span & true_span)
                if overlap == 0:
                    f1 = 0.0
                else:
                    precision = overlap / len(pred_span)
                    recall = overlap / len(true_span)
                    f1 = 2 * (precision * recall) / (precision + recall)

                total_f1 += f1
            total_samples += len(batch['input_ids'])

    avg_em = total_em / total_samples
    avg_f1 = total_f1 / total_samples

    print(f"Exact Match: {avg_em:.4f}")
    print(f"F1 Score: {avg_f1:.4f}")

    return avg_em, avg_f1


In [ ]:
evaluate_full_answer_span(model, test_loader, tokenizer, device)

TypeError: HierarchicalBinaryTree.forward() missing 2 required positional arguments: 'label_pos' and 'teacher_attn'

In [ ]:
for batch in test_loader:
  break
batch.keys()

dict_keys(['input_ids', 'attention_mask', 'label_pos'])

In [ ]:
batch['label_pos']

tensor([0, 0, 0, 0, 0, 0, 0, 0])

In [ ]:
batch['input_ids']

tensor([[ 101,  101, 1249,  ...,    0,    0,    0],
        [ 101,  101, 1335,  ...,    0,    0,    0],
        [ 101,  101, 4827,  ...,    0,    0,    0],
        ...,
        [ 101,  101, 1130,  ...,    0,    0,    0],
        [ 101,  101, 2627,  ...,    0,    0,    0],
        [ 101,  101, 1130,  ...,    0,    0,    0]])

In [ ]:
batch['attention_mask']

tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])

In [ ]:
test_data[0].keys()

dict_keys(['question', 'question_id', 'question_source', 'entity_pages', 'search_results', 'answer'])

In [ ]:
test_data[0]['answer']

{'aliases': [],
 'normalized_aliases': [],
 'matched_wiki_entity_name': '<unk>',
 'normalized_matched_wiki_entity_name': '<unk>',
 'normalized_value': '<unk>',
 'type': '',
 'value': '<unk>'}